In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 1. CARREGAMENTO E CURADORIA DE DADOS
# Justificativa: Utilizamos metadados do Scopus (Títulos e Resumos).
# O tratamento de nulos é ético: evita viés de sub-representação de autores.
df = pd.read_csv('../data/scopus_metadata.csv').dropna(subset=['Abstract'])

# 2. VETORIZAÇÃO (TF-IDF)
# Por que TF-IDF? Diferente do CountVectorizer, ele penaliza termos muito comuns 
# na academia (ex: "research", "paper"), destacando termos de real semântica latente.
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2)
tfidf_matrix = vectorizer.fit_transform(df['Abstract'])

# 3. MODELAGEM DE TÓPICOS (LDA)
# Decisão Metodológica: Definimos K=5 tópicos para garantir a interpretabilidade humana.
# Um K muito alto (ex: 50) geraria fragmentação e perda de governança sobre o sentido.
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)
lda_topics = lda_model.fit_transform(tfidf_matrix)

# 4. EXPLICABILIDADE DOS RESULTADOS
# Função que mapeia os termos mais importantes por tópico para validação do especialista.
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Tópico {topic_idx}: " + " | ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

# Acionamento da transparência algorítmica
display_topics(lda_model, vectorizer.get_feature_names_out(), 10)